# Installing Dependencies

In [1]:
%pip install lightgbm xgboost scikit-learn numpy pandas scipy


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

TRAIN_LABEL = train_labels
TEST_DATA   = test_labels

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)


Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)


## 1. Session Baselines + Dead EDA Detection

In [3]:
def compute_subject_baseline(sensor_df, val_col):
    out = {}
    for pid, grp in sensor_df.groupby('pid'):
        v = grp[val_col].dropna()
        out[pid] = (v.mean(), v.std() + 1e-8)
    return out

def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
            print(f'  {pid}: EDA zero ratio={zr:.1%} → DEAD')
    return dead

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

print('Train EDA dead sensors:')
TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
print('Test EDA dead sensors:')
TEST_EDA_DEAD  = eda_dead_subjects(testeda)
print(f'Train dead: {TRAIN_EDA_DEAD} | Test dead: {TEST_EDA_DEAD}')


Train EDA dead sensors:
  70N8: EDA zero ratio=99.6% → DEAD
  Y21H: EDA zero ratio=99.4% → DEAD
Test EDA dead sensors:
Train dead: {'Y21H', '70N8'} | Test dead: set()


## 2. Feature Extraction (v27 — IBI/ACC extended, 15 s window added)

In [4]:
# v27 changes vs v18:
#   • WINDOWS_MS expanded: added 15 s window for slow-changing signals
#   • IBI block: +skew, +kurt, +q25, +q75, +iqr, +pnn50  (top features in fixed pipeline)
#   • ACC block: +max, +q25, +q75, +iqr                   (acc_max/iqr in fixed top-20)
#   • Count features intentionally NOT computed (data-density proxy, not physiology)

from scipy.stats import skew as _sp_skew, kurtosis as _sp_kurt

WINDOWS_MS  = [2500, 5000, 10000, 15000]   # added 15 s
ROLL_WIN_MS = 30000


def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
    else:
        for s in ['mean', 'std', 'range', 'slope', 'p25', 'p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def ibi_extended(ibi_v, prefix, bl_m, bl_s):
    """Full HRV feature set for a window of IBI values."""
    feat = {}
    n = len(ibi_v)

    # Basic stats (need >= 2 samples)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(ibi_v)
        feat[f'{prefix}_std']   = np.std(ibi_v)
        feat[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(ibi_v)**2))
        feat[f'{prefix}_dev']   = (np.mean(ibi_v) - bl_m) / bl_s
        feat[f'{prefix}_q25']   = np.percentile(ibi_v, 25)
        feat[f'{prefix}_q75']   = np.percentile(ibi_v, 75)
        feat[f'{prefix}_iqr']   = feat[f'{prefix}_q75'] - feat[f'{prefix}_q25']
        feat[f'{prefix}_range'] = np.max(ibi_v) - np.min(ibi_v)
        diffs = np.abs(np.diff(ibi_v))
        feat[f'{prefix}_pnn50'] = np.mean(diffs > 50) if len(diffs) > 0 else np.nan
    else:
        for s in ['mean', 'std', 'rmssd', 'dev', 'q25', 'q75', 'iqr', 'range', 'pnn50']:
            feat[f'{prefix}_{s}'] = np.nan

    # Skewness & kurtosis (need >= 3 samples)
    if n >= 3:
        feat[f'{prefix}_skew'] = float(_sp_skew(ibi_v))
        feat[f'{prefix}_kurt'] = float(_sp_kurt(ibi_v))
    else:
        feat[f'{prefix}_skew'] = np.nan
        feat[f'{prefix}_kurt'] = np.nan

    return feat


def acc_extended(acc_v, prefix, bl_m, bl_s):
    """Extended accelerometer features for a window."""
    feat = {}
    n = len(acc_v)
    if n >= 5:
        feat[f'{prefix}_mean']   = np.mean(acc_v)
        feat[f'{prefix}_std']    = np.std(acc_v)
        feat[f'{prefix}_energy'] = np.mean(acc_v**2)
        feat[f'{prefix}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
        feat[f'{prefix}_max']    = np.max(acc_v)
        feat[f'{prefix}_q25']    = np.percentile(acc_v, 25)
        feat[f'{prefix}_q75']    = np.percentile(acc_v, 75)
        feat[f'{prefix}_iqr']    = feat[f'{prefix}_q75'] - feat[f'{prefix}_q25']
    else:
        for s in ['mean', 'std', 'energy', 'dev', 'max', 'q25', 'q75', 'iqr']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def nan_eda_features(feat, wl):
    for key in list(feat.keys()):
        if f'eda_{wl}' in key and key not in [f'eda_{wl}_valid', f'eda_{wl}_zero_ratio']:
            feat[key] = np.nan
    return feat


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp,
                         sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid', 'timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        feat['bl_hr']   = bl_hr.get(pid,   (np.nan, 1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan, 1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan, 1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan, 1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan, 1))[0]

        def get_win(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            # ── HR ──────────────────────────────────────────────────────────
            hr_v = get_win(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan

            # ── EDA (with NaN masking for dead sensors) ──────────────────────
            eda_v = get_win(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
                feat[f'eda_{wl}_nz_frac'] = len(nz) / len(eda_v)
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, wl)
            else:
                for k in ['zero_ratio', 'valid', 'dev', 'nz_mean', 'nz_frac']:
                    feat[f'eda_{wl}_{k}'] = np.nan
                feat = nan_eda_features(feat, wl)

            # ── TEMP ─────────────────────────────────────────────────────────
            temp_v = get_win(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev'] = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan

            # ── IBI (v27: extended HRV features) ─────────────────────────────
            ibi_v = get_win(ibi_df, 'value', hw)
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            feat.update(ibi_extended(ibi_v, f'ibi_{wl}', bl_m, bl_s))

            # ── ACC (v27: extended features) ──────────────────────────────────
            acc_v = get_win(acc_df, 'magnitude', hw)
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            feat.update(acc_extended(acc_v, f'acc_{wl}', bl_m, bl_s))

            # ── BVP ──────────────────────────────────────────────────────────
            bvp_v = get_win(bvp_df, 'value', hw)
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v, 75) - np.percentile(bvp_v, 25)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
            else:
                for s in ['std', 'range', 'iqr', 'dev']:
                    feat[f'bvp_{wl}_{s}'] = np.nan

            # ── EEG ──────────────────────────────────────────────────────────
            s_eeg   = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts - hw) & (s_eeg.timestamp < ts + hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = win_eeg[col].mean()
                th = feat[f'eeg_theta_{wl}']
                la = feat[f'eeg_lowAlpha_{wl}']
                ha = feat[f'eeg_highAlpha_{wl}']
                lb = feat[f'eeg_lowBeta_{wl}']
                hb = feat[f'eeg_highBeta_{wl}']
                lg = feat[f'eeg_lowGamma_{wl}']
                de = feat[f'eeg_delta_{wl}']
                feat[f'eeg_theta_alpha_{wl}']  = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wl}']   = (lb + hb) / (la + ha + eps)
                feat[f'eeg_hbeta_lgamma_{wl}'] = hb / (lg + eps)
                feat[f'eeg_engage_{wl}']        = hb / (de + th + eps)
            else:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = np.nan
                for r in ['theta_alpha', 'beta_alpha', 'hbeta_lgamma', 'engage']:
                    feat[f'eeg_{r}_{wl}'] = np.nan

        # ── Rolling deviation (30 s look-back) ───────────────────────────────
        for sensor, df_, col in [('hr', hr_df, 'value'), ('temp', temp_df, 'value')]:
            s    = df_[df_.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)][col].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)][col].values
            if len(past) >= 2 and len(cur) >= 1:
                feat[f'{sensor}_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat[f'{sensor}_roll_dev'] = np.nan

        if pid not in eda_dead_set:
            s    = eda_df[eda_df.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)]['value'].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)]['value'].values
            if len(past) >= 2 and len(cur) >= 1:
                feat['eda_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat['eda_roll_dev'] = np.nan
        else:
            feat['eda_roll_dev'] = np.nan

        # ── Interaction features ──────────────────────────────────────────────
        hr_m  = feat.get('hr_w5s_mean',  np.nan)
        tmp_m = feat.get('temp_w5s_mean', np.nan)
        hr_d  = feat.get('hr_w5s_dev',   np.nan)
        eda_d = feat.get('eda_w5s_dev',  np.nan)
        feat['hr_temp_product']    = hr_m * tmp_m
        feat['dev_hr_eda_product'] = hr_d * eda_d

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction ready.')


Feature extraction ready.


In [5]:
print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)


Extracting TRAIN features...
Train features: (1456, 256)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)


Extracting TEST features...
Test features: (1496, 255)


## 3. Lag Features

In [7]:
# v27: IBI mean/std added to lag columns
LAG_BASE = [
    'hr_w5s_mean',   'hr_w5s_dev',   'hr_roll_dev',
    'eda_w5s_mean',  'eda_w5s_dev',  'eda_roll_dev',
    'temp_w5s_mean', 'temp_w5s_dev', 'temp_roll_dev',
    'bvp_w5s_std',
    'ibi_w5s_mean',  'ibi_w5s_std',   # v27 addition
]
LAG_COLS = [c for c in LAG_BASE if c in train_feats.columns]


def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid', 'timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    for col in cols:
        df[f'{col}_roll3'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    return df


train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id', 'pid', 'timestamp', 'arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]
print(f'Total features: {len(FEAT_COLS)}')


Total features: 288


## 4. Training Setup

In [8]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, classification_report

train_feats_sorted = train_feats.sort_values(['pid', 'timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_all)
TRAIN_PRIOR = np.bincount(y_all, minlength=5) / len(y_all)
print('Class weights:', {f'A{i+1}': round(w, 2) for i, w in enumerate(cw)})
print('Training prior:', {f'A{i+1}': round(p, 3) for i, p in enumerate(TRAIN_PRIOR)})


Class weights: {'A1': np.float64(5.29), 'A2': np.float64(0.68), 'A3': np.float64(0.53), 'A4': np.float64(0.84), 'A5': np.float64(4.04)}
Training prior: {'A1': np.float64(0.038), 'A2': np.float64(0.295), 'A3': np.float64(0.38), 'A4': np.float64(0.237), 'A5': np.float64(0.049)}


## 5. LOSO CV

In [9]:
import lightgbm as lgb
import xgboost as xgb

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS      = [42, 7, 123, 13, 99]

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_xgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float64)
loso_lgb, loso_xgb = [], []


def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=63, learning_rate=0.03,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=15, lambda_l1=0.3, lambda_l2=0.3,
        max_depth=7, verbose=-1, seed=seed, n_jobs=-1
    )


def get_xgb_params(seed):
    return dict(
        objective='multi:softprob', num_class=5, eval_metric='mlogloss',
        max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7,
        min_child_weight=10, reg_alpha=0.3, reg_lambda=0.3,
        seed=seed, verbosity=0, nthread=-1
    )


for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    sw_tr = cw[y_tr]

    f_lgb = np.zeros((va_mask.sum(), 5))
    f_xgb = np.zeros((va_mask.sum(), 5))
    t_lgb = np.zeros((len(test_feats), 5))
    t_xgb = np.zeros((len(test_feats), 5))

    for seed in SEEDS:
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr, num_boost_round=1500,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        f_lgb += m.predict(X_va)   / len(SEEDS)
        t_lgb += m.predict(X_test) / len(SEEDS)

        dtr_x = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
        dva_x = xgb.DMatrix(X_va, label=y_va)
        m_x = xgb.train(
            get_xgb_params(seed), dtr_x, num_boost_round=1500,
            evals=[(dva_x, 'val')],
            early_stopping_rounds=100, verbose_eval=False,
        )
        f_xgb += m_x.predict(dva_x).reshape(-1, 5)               / len(SEEDS)
        t_xgb += m_x.predict(xgb.DMatrix(X_test)).reshape(-1, 5) / len(SEEDS)

    oof_lgb[va_mask] = f_lgb
    oof_xgb[va_mask] = f_xgb
    test_lgb += t_lgb / len(TRAIN_PIDS)
    test_xgb += t_xgb / len(TRAIN_PIDS)

    ba_l = balanced_accuracy_score(y_va, f_lgb.argmax(axis=1))
    ba_x = balanced_accuracy_score(y_va, f_xgb.argmax(axis=1))
    loso_lgb.append(ba_l)
    loso_xgb.append(ba_x)
    print(f'  {fold_pid} — LGB: {ba_l:.4f} | XGB: {ba_x:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_lgb):.4f} ± {np.std(loso_lgb):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb):.4f} ± {np.std(loso_xgb):.4f}')


  01Z2 — LGB: 0.3222 | XGB: 0.2444
  70N8 — LGB: 0.1938 | XGB: 0.2378
  7PF3 — LGB: 0.2073 | XGB: 0.1776
  CQ2G — LGB: 0.2063 | XGB: 0.2747
  D1XP — LGB: 0.2715 | XGB: 0.2512
  DT5C — LGB: 0.2601 | XGB: 0.2375
  F1ZM — LGB: 0.1427 | XGB: 0.1402
  LIUY — LGB: 0.4939 | XGB: 0.5679
  SE4Q — LGB: 0.2927 | XGB: 0.5228
  TPQI — LGB: 0.0391 | XGB: 0.0809
  Y21H — LGB: 0.1877 | XGB: 0.1079

LOSO LGB mean: 0.2379 ± 0.1096
LOSO XGB mean: 0.2585 ± 0.1483


## 6. Blend + Per-class Threshold Optimisation (v27)

In [10]:
from scipy.optimize import minimize

# ── Step 1: find best LGB/XGB blend weight ──────────────────────────────────
best_ba_raw, best_w = 0.0, 0.5
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgb + (1 - w) * oof_xgb
    ba    = balanced_accuracy_score(y_all, blend.argmax(axis=1))
    if ba > best_ba_raw:
        best_ba_raw, best_w = ba, w

print(f'Best blend LGB weight: {best_w:.2f}  →  OOF BA (argmax): {best_ba_raw:.4f}')

oof_blend  = best_w * oof_lgb  + (1 - best_w) * oof_xgb
test_blend = best_w * test_lgb + (1 - best_w) * test_xgb


# ── Step 2: per-class threshold optimisation on OOF probs ───────────────────
# Multiply each class column by exp(t_i) before argmax.
# This up-/down-weights rare classes (1 and 5) to equalise recall.

def apply_thresholds(t, probs):
    return (probs * np.exp(t)).argmax(axis=1)

def neg_ba(t, probs, y_true):
    return -balanced_accuracy_score(y_true, apply_thresholds(t, probs))

# Random search for a good starting point
rng = np.random.default_rng(42)
best_ba_thr, best_thr = best_ba_raw, np.zeros(5)

for _ in range(8000):
    t = rng.uniform(-2.5, 2.5, 5)
    ba = -neg_ba(t, oof_blend, y_all)
    if ba > best_ba_thr:
        best_ba_thr, best_thr = ba, t.copy()

# Refine with Nelder-Mead
res = minimize(neg_ba, best_thr, args=(oof_blend, y_all),
               method='Nelder-Mead',
               options={'maxiter': 20000, 'xatol': 1e-6, 'fatol': 1e-6})
if -res.fun > best_ba_thr:
    best_thr    = res.x
    best_ba_thr = -res.fun

print(f'OOF BA before threshold opt: {best_ba_raw:.4f}')
print(f'OOF BA after  threshold opt: {best_ba_thr:.4f}')
print(f'Optimal log-scale class weights: {best_thr.round(4)}')

# Apply to test predictions
test_pred = apply_thresholds(best_thr, test_blend) + 1   # back to 1-based

print('\nTest distribution:')
print(pd.Series(test_pred).value_counts().sort_index())
print('Expected from prior:', {f'A{i+1}': int(p * len(test_pred)) for i, p in enumerate(TRAIN_PRIOR)})


Best blend LGB weight: 1.00  →  OOF BA (argmax): 0.2127
OOF BA before threshold opt: 0.2127
OOF BA after  threshold opt: 0.3750
Optimal log-scale class weights: [ 0.6618 -0.5261 -0.5881  0.2433 -1.2454]

Test distribution:
1    439
2     97
3    240
4    719
5      1
Name: count, dtype: int64
Expected from prior: {'A1': 56, 'A2': 441, 'A3': 569, 'A4': 354, 'A5': 73}


## 7. OOF Diagnosis

In [11]:
oof_pred_raw = oof_blend.argmax(axis=1)
oof_pred_thr = apply_thresholds(best_thr, oof_blend)

print('=== OOF Classification Report (threshold-optimised) ===')
print(classification_report(y_all, oof_pred_thr,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print('Per-subject LOSO BA:')
for pid, ba_l, ba_x in zip(TRAIN_PIDS, loso_lgb, loso_xgb):
    flag = ' ← LOW' if max(ba_l, ba_x) < 0.25 else ''
    print(f'  {pid}: LGB={ba_l:.4f} XGB={ba_x:.4f}{flag}')

print(f'\nFinal OOF BA (argmax)    : {balanced_accuracy_score(y_all, oof_pred_raw):.4f}')
print(f'Final OOF BA (thr-opt)   : {balanced_accuracy_score(y_all, oof_pred_thr):.4f}')
print(f'LOSO LGB mean            : {np.mean(loso_lgb):.4f}')
print(f'LOSO XGB mean            : {np.mean(loso_xgb):.4f}')


=== OOF Classification Report (threshold-optimised) ===
              precision    recall  f1-score   support

   Arousal 1       0.12      0.89      0.22        55
   Arousal 2       0.39      0.21      0.27       430
   Arousal 3       0.48      0.19      0.28       554
   Arousal 4       0.34      0.58      0.43       345
   Arousal 5       0.00      0.00      0.00        72

    accuracy                           0.31      1456
   macro avg       0.26      0.37      0.24      1456
weighted avg       0.38      0.31      0.29      1456

Per-subject LOSO BA:
  01Z2: LGB=0.3222 XGB=0.2444
  70N8: LGB=0.1938 XGB=0.2378 ← LOW
  7PF3: LGB=0.2073 XGB=0.1776 ← LOW
  CQ2G: LGB=0.2063 XGB=0.2747
  D1XP: LGB=0.2715 XGB=0.2512
  DT5C: LGB=0.2601 XGB=0.2375
  F1ZM: LGB=0.1427 XGB=0.1402 ← LOW
  LIUY: LGB=0.4939 XGB=0.5679
  SE4Q: LGB=0.2927 XGB=0.5228
  TPQI: LGB=0.0391 XGB=0.0809 ← LOW
  Y21H: LGB=0.1877 XGB=0.1079 ← LOW

Final OOF BA (argmax)    : 0.2127
Final OOF BA (thr-opt)   : 0.3750
LOSO 

# Generating Final Submissions

In [12]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred,
})

submission.to_csv('submission-v27.csv', index=False)
print('submission-v27.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())


submission-v27.csv saved.
Shape: (1496, 2)
arousal
1    439
2     97
3    240
4    719
5      1
Name: count, dtype: int64
